# Gradient Boosting & AdaBoost from Scratch

Two boosting algorithms built with plain numpy:

1. **Gradient Boosting Regressor** — sequentially fits shallow regression trees to the residual errors of the ensemble so far, shrinking each tree's contribution by a learning rate.
2. **AdaBoost Classifier** — sequentially fits decision stumps, reweighting misclassified samples after each round and combining stumps via a weighted vote.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

np.random.seed(42)

## Part 1: Gradient Boosting Regressor

A shallow CART regression tree (splits chosen by MSE reduction) is the weak learner. Each boosting round fits a new tree to the current residual and adds `learning_rate * tree.predict(X)` to the running prediction.

In [ ]:
class _Node:
    __slots__ = ('feat', 'thresh', 'left', 'right', 'value')
    def __init__(self, feat=None, thresh=None, left=None, right=None, value=None):
        self.feat, self.thresh, self.left, self.right, self.value = feat, thresh, left, right, value


def _best_split_mse(X, y):
    n, d = X.shape
    best_feat, best_thresh, best_mse = None, None, np.inf
    for feat in range(d):
        vals = np.unique(X[:, feat])
        if len(vals) < 2:
            continue
        thresholds = (vals[:-1] + vals[1:]) / 2
        for t in thresholds:
            left_mask = X[:, feat] <= t
            if left_mask.sum() == 0 or left_mask.sum() == n:
                continue
            y_l, y_r = y[left_mask], y[~left_mask]
            mse = np.var(y_l) * len(y_l) + np.var(y_r) * len(y_r)
            if mse < best_mse:
                best_mse, best_feat, best_thresh = mse, feat, t
    return best_feat, best_thresh

In [ ]:
class RegressionTree:
    """Shallow CART tree used as the weak learner inside boosting."""

    def __init__(self, max_depth=3, min_samples_split=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.root = self._grow(X, y, depth=0)
        return self

    def _grow(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or np.var(y) == 0:
            return _Node(value=np.mean(y))
        feat, thresh = _best_split_mse(X, y)
        if feat is None:
            return _Node(value=np.mean(y))
        mask = X[:, feat] <= thresh
        left = self._grow(X[mask], y[mask], depth + 1)
        right = self._grow(X[~mask], y[~mask], depth + 1)
        return _Node(feat=feat, thresh=thresh, left=left, right=right)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feat] <= node.thresh:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

In [ ]:
class GradientBoostingRegressorScratch:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    def fit(self, X, y):
        self.init_value = np.mean(y)
        residual = y - self.init_value
        self.trees = []
        for _ in range(self.n_estimators):
            tree = RegressionTree(max_depth=self.max_depth).fit(X, residual)
            residual = residual - self.learning_rate * tree.predict(X)
            self.trees.append(tree)
        return self

    def staged_predict(self, X):
        pred = np.full(X.shape[0], self.init_value, dtype=float)
        for tree in self.trees:
            pred = pred + self.learning_rate * tree.predict(X)
            yield pred.copy()

    def predict(self, X):
        pred = np.full(X.shape[0], self.init_value, dtype=float)
        for tree in self.trees:
            pred = pred + self.learning_rate * tree.predict(X)
        return pred

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingRegressorScratch(n_estimators=100, learning_rate=0.1, max_depth=3).fit(X_tr, y_tr)
pred_te = gb.predict(X_te)

sk_gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_tr, y_tr)

print("=" * 46)
print(f"{'Model':<28}  {'Test MSE':>12}")
print("=" * 46)
print(f"{'Scratch GradientBoosting':<28}  {mean_squared_error(y_te, pred_te):>12.2f}")
print(f"{'sklearn GradientBoosting':<28}  {mean_squared_error(y_te, sk_gb.predict(X_te)):>12.2f}")
print("=" * 46)

In [ ]:
# Staged MSE — shows the ensemble converging as trees are added
staged_train_mse = [mean_squared_error(y_tr, p) for p in gb.staged_predict(X_tr)]
staged_test_mse = [mean_squared_error(y_te, p) for p in gb.staged_predict(X_te)]

plt.figure(figsize=(7, 4))
plt.plot(staged_train_mse, label='Train MSE')
plt.plot(staged_test_mse, label='Test MSE')
plt.xlabel('Boosting round'); plt.ylabel('MSE')
plt.title('Gradient Boosting — MSE vs. Number of Trees')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Learning-rate sweep: too small underfits in 100 rounds, too large overshoots
rates = [0.01, 0.05, 0.1, 0.3, 1.0]
rate_mse = []
for lr in rates:
    model = GradientBoostingRegressorScratch(n_estimators=100, learning_rate=lr, max_depth=3).fit(X_tr, y_tr)
    rate_mse.append(mean_squared_error(y_te, model.predict(X_te)))

plt.figure(figsize=(7, 4))
plt.plot(rates, rate_mse, marker='o')
plt.xscale('log')
plt.xlabel('Learning rate (log scale)'); plt.ylabel('Test MSE')
plt.title('Gradient Boosting — Learning Rate Sweep (100 trees)')
plt.tight_layout(); plt.show()

for lr, mse in zip(rates, rate_mse):
    print(f"lr={lr:<5}  test MSE={mse:.2f}")

## Part 2: AdaBoost Classifier

The weak learner is a decision stump (a single-feature, single-threshold split). After each round, misclassified samples get more weight so the next stump focuses on them; each stump's vote is weighted by `alpha = 0.5 * ln((1 - err) / err)`.

In [ ]:
class DecisionStump:
    """Weighted single-split classifier: predicts +/-1 based on one feature threshold."""

    def fit(self, X, y, sample_weight):
        n, d = X.shape
        best_err = np.inf
        for feat in range(d):
            vals = np.unique(X[:, feat])
            thresholds = (vals[:-1] + vals[1:]) / 2 if len(vals) > 1 else vals
            for t in thresholds:
                for polarity in (1, -1):
                    pred = np.where(polarity * X[:, feat] <= polarity * t, 1, -1)
                    err = np.sum(sample_weight[pred != y])
                    if err < best_err:
                        best_err = err
                        self.feat, self.thresh, self.polarity = feat, t, polarity
        return best_err

    def predict(self, X):
        return np.where(self.polarity * X[:, self.feat] <= self.polarity * self.thresh, 1, -1)

In [ ]:
class AdaBoostScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators

    def fit(self, X, y):
        n = X.shape[0]
        w = np.full(n, 1 / n)
        self.stumps, self.alphas = [], []
        for _ in range(self.n_estimators):
            stump = DecisionStump()
            err = np.clip(stump.fit(X, y, w), 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - err) / err)
            pred = stump.predict(X)
            w = w * np.exp(-alpha * y * pred)
            w = w / w.sum()
            self.stumps.append(stump)
            self.alphas.append(alpha)
        return self

    def staged_predict(self, X):
        score = np.zeros(X.shape[0])
        for alpha, stump in zip(self.alphas, self.stumps):
            score = score + alpha * stump.predict(X)
            yield np.sign(score)

    def decision_function(self, X):
        return sum(a * s.predict(X) for a, s in zip(self.alphas, self.stumps))

    def predict(self, X):
        return np.sign(self.decision_function(X))

In [ ]:
Xc, yc_raw = load_breast_cancer(return_X_y=True)
yc = np.where(yc_raw == 0, -1, 1)   # AdaBoost expects labels in {-1, +1}
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42)

ada = AdaBoostScratch(n_estimators=50).fit(Xc_tr, yc_tr)
sk_ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42).fit(Xc_tr, yc_tr)

print("=" * 46)
print(f"{'Model':<28}  {'Test Accuracy':>12}")
print("=" * 46)
print(f"{'Scratch AdaBoost':<28}  {accuracy_score(yc_te, ada.predict(Xc_te)):>12.4f}")
print(f"{'sklearn AdaBoost':<28}  {accuracy_score(yc_te, sk_ada.predict(Xc_te)):>12.4f}")
print("=" * 46)

In [ ]:
# Accuracy as stumps are added, and the alpha (vote weight) each stump earned
staged_acc = [accuracy_score(yc_te, p) for p in ada.staged_predict(Xc_te)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(staged_acc)
ax1.set_xlabel('Boosting round'); ax1.set_ylabel('Test accuracy')
ax1.set_title('AdaBoost — Accuracy vs. Number of Stumps')

ax2.plot(ada.alphas, color='tab:orange')
ax2.set_xlabel('Boosting round'); ax2.set_ylabel('alpha (stump weight)')
ax2.set_title('AdaBoost — Vote Weight per Stump')

plt.tight_layout(); plt.show()